In [ ]:
# mount and install
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q neuralforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.0/308.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 64.3 MB/s eta 0:00:00


In [ ]:
import sys, os
sys.path.append('/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/src')

import numpy as np
import pandas as pd
import config as cf, data, predictions

if os.path.exists(cf.MODEL_INPUT):
    df = data.load()
else:
    df = data.build()

print(len(df), 'rows')

4850000 rows


In [ ]:
static_df=df[['unique_id'] + cf.STATIC].drop_duplicates('unique_id').reset_index(drop=True)
print(static_df.shape)      # (50000, 6)

(50000, 6)


In [ ]:
# common harness for all models
from neuralforecast import NeuralForecast

def run_neural(make_model, name, train_df, futr_df, input_version, split,
               covariate=True, static=True):

    model = make_model()
    cols = ['unique_id', 'ds', 'y'] + (cf.EXOG if covariate else [])

    nf = NeuralForecast(models=[model], freq='D')
    nf.fit(train_df[cols], static_df=static_df if static else None)

    preds = nf.predict(futr_df=futr_df) if covariate else nf.predict()
    if 'unique_id' not in preds.columns:
        preds = preds.reset_index()


    col = model.__class__.__name__
    p = preds[['unique_id', 'ds', col]].rename(columns={col: 'prediction'})
    p[['store_id', 'product_id']] = p['unique_id'].str.split('_', expand=True).astype(int)
    p['dt'] = pd.to_datetime(p['ds']).dt.strftime('%Y-%m-%d')

    predictions.save(p[['store_id', 'product_id', 'dt', 'prediction']],
                     name, input_version, split)


def run_all(make_model, name, covariate=True, static=True):
    """All four passes for one model."""
    for input_version in ['raw', 'recovered']:
        for split in ['val', 'test']:
            train_df, futr_df = data.splittbl(df, input_version, split)
            print(f'=== {name} | {input_version} | {split} ===')
            run_neural(make_model, name, train_df, futr_df, input_version, split,
                       covariate=covariate, static=static)

In [ ]:
# TFT
from neuralforecast.models import TFT

run_all(lambda: TFT(h=cf.HORIZON, input_size=cf.INPUT_SIZE, max_steps=cf.MAX_STEPS,
                    scaler_type=cf.SCALER,
                    futr_exog_list=cf.EXOG, stat_exog_list=cf.STATIC),
        'TFT')

INFO:lightning_fabric.utilities.seed:Seed set to 1


=== TFT | raw | val ===


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 3.3 K  | train
7  | static_encoder          | StaticCov

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

[test] 350,000 rows : TFT__recovered


In [ ]:
# TiDE (covariates + statics, like TFT but simpler)
from neuralforecast.models import TiDE

run_all(lambda: TiDE(h=cf.HORIZON, input_size=cf.INPUT_SIZE, max_steps=cf.MAX_STEPS,
                     scaler_type=cf.SCALER,
                     futr_exog_list=cf.EXOG, stat_exog_list=cf.STATIC),
        'TiDE')

INFO:lightning_fabric.utilities.seed:Seed set to 1


=== TiDE | raw | val ===


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:124: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                 | Type          | Params | Mode 
----------------------------------------------------------------
0  | loss                 | MAE           | 0      | train
1  | hist_cat_embeddings  | ModuleList    | 0      | train
2  | futr_cat_embeddings  | ModuleList    | 0      | train
3  | stat_cat_embeddings  | ModuleList    | 0      | train
4  | padder_train         | ConstantPad1d | 0      | train
5  | scaler               | TemporalNorm  | 0      | train
6  | hist_exog_projection | MLPResidual   

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

[test] 350,000 rows : TiDE__recovered


In [ ]:
# LSTM (covariates + statics)
from neuralforecast.models import LSTM

run_all(lambda: LSTM(h=cf.HORIZON, input_size=cf.INPUT_SIZE, max_steps=cf.MAX_STEPS,
                     scaler_type=cf.SCALER,
                     futr_exog_list=cf.EXOG, stat_exog_list=cf.STATIC),
        'LSTM')

INFO:lightning_fabric.utilities.seed:Seed set to 1


=== LSTM | raw | val ===


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | hist_encoder        | LSTM          | 205 K  | train
7 | mlp_decoder         | MLP           | 17.5 K | train
--------------------------------------------------------------
222 K     Trainable params
0         Non-trainable params
222 K 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

[test] 350,000 rows : LSTM__recovered


In [ ]:
# Informer
from neuralforecast.models import Informer

run_all(lambda: Informer(h=cf.HORIZON, input_size=cf.INPUT_SIZE, max_steps=cf.MAX_STEPS,
                         scaler_type=cf.SCALER, futr_exog_list=cf.EXOG),
        'Informer', static=False)

INFO:lightning_fabric.utilities.seed:Seed set to 1


=== Informer | raw | val ===


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | enc_embedding       | DataEmbedding | 1.3 K  | train
7 | dec_embedding       | DataEmbedding | 1.3 K  | train
8 | encoder             | TransEncoder  | 199 K  | train
9 | decoder             | TransDecoder  | 141 K  | train
-------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

[test] 350,000 rows : Informer__recovered


In [ ]:
# DLinear
from neuralforecast.models import DLinear

run_all(lambda: DLinear(h=cf.HORIZON, input_size=cf.INPUT_SIZE, max_steps=cf.MAX_STEPS,
                        scaler_type=cf.SCALER),
        'DLinear', covariate=False, static=False)

INFO:lightning_fabric.utilities.seed:Seed set to 1


=== DLinear | raw | val ===


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | decomp              | SeriesDecomp  | 0      | train
7 | linear_trend        | Linear        | 427    | train
8 | linear_season       | Linear        | 427    | train
--------------------------------------------------------------
854    

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

[test] 350,000 rows : DLinear__recovered


In [ ]:
# PatchTST
from neuralforecast.models import PatchTST

run_all(lambda: PatchTST(h=cf.HORIZON, input_size=cf.INPUT_SIZE, max_steps=cf.MAX_STEPS,
                         scaler_type=cf.SCALER),
        'PatchTST', covariate=False, static=False)

INFO:lightning_fabric.utilities.seed:Seed set to 1


=== PatchTST | raw | val ===


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 406 K  | train
------------------------------------------------------------------
406 K     Trainable params
3         Non-trainable params
406 K     Total params


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

[test] 350,000 rows : PatchTST__recovered


In [ ]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# environment record
import platform, subprocess, sys
from importlib.metadata import version, PackageNotFoundError

print("OS      :", platform.platform())
print("Python  :", sys.version.split()[0])

try:
    import torch
    print("torch   :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU     :", torch.cuda.get_device_name(0),
              "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
except ImportError:
    print("torch   : not installed")

print("CPU     :", subprocess.run("nproc", capture_output=True, text=True).stdout.strip(), "cores")
print("RAM     :", subprocess.run("free -g | awk 'NR==2{print $2}'", shell=True,
                                  capture_output=True, text=True).stdout.strip(), "GB")

PKGS = ["neuralforecast", "mlforecast", "lightgbm", "statsforecast", "optuna",
        "timesfm", "chronos-forecasting", "pypots", "pandas", "numpy", "pyarrow",
        "scikit-learn", "pytorch-lightning"]
print()
for p in PKGS:
    try:
        print(f"{p:22s} {version(p)}")
    except PackageNotFoundError:
        print(f"{p:22s} -")

OS      : Linux-6.6.122+-x86_64-with-glibc2.35
Python  : 3.12.13
torch   : 2.11.0+cu128 | CUDA available: True
GPU     : Tesla T4 | 15.6 GB
CPU     : 8 cores
RAM     : 50 GB

neuralforecast         3.2.1
mlforecast             -
lightgbm               4.6.0
statsforecast          -
optuna                 4.9.0
timesfm                -
chronos-forecasting    -
pypots                 -
pandas                 2.2.2
numpy                  2.0.2
pyarrow                18.1.0
scikit-learn           1.6.1
pytorch-lightning      2.5.6
